# 02 · 실시간 졸음 감지 — YuNet + Eye/Yawn CNN

학습된 가중치를 **불러와서 쓰기만** 합니다. 위에서 아래로 실행하면 마지막 셀에서
웹캠 실시간 모니터가 뜹니다.

## 하품 판정은 3단계다

프레임 한 장으로는 하품과 말하기를 가를 수 없습니다. 어느 한 순간만 보면 둘 다 그냥
"입이 벌어진 얼굴"입니다. 그래서 판정을 셋으로 나눕니다.

| 단계 | 누가 | 무엇을 거르나 |
|---|---|---|
| 1. 게이트 | 랜드마크 | 입을 다문 프레임. CNN 을 아예 부르지 않는다 |
| 2. 판정 | **CNN** | 입은 벌렸다 — 하품인가 말하기인가 |
| 3. 누적 | 시간 | 순간적으로 튄 오판 |

1번과 3번은 CNN 이 구조적으로 못 하는 일입니다. 학습 데이터에 입 다문 프레임이 아예
없고(그런 입력은 출력이 무의미합니다), CNN 은 한 장만 보므로 시간을 모릅니다.

말하기 오경보가 0.683 → 0.195 로 줄고, 손 안 가린 하품 검출은 1.000 을 유지합니다.

## 필요한 것

```
model/artifacts/eye_mrl+dmd__eval-dmd__gray128.keras
model/artifacts/yawn_yawn_mouthopen_v2__zoo-cnn_large__eval-face__gray128.keras
model/artifacts/*_metrics.json                       <- 저장소에 이미 있음
model/detectors/face_detection_yunet_2023mar.onnx    <- 저장소에 이미 있음
model/detectors/face_landmarker.task                 <- 게이트용. 직접 받아야 함
```

`.keras` 두 개는 Release 에서 받습니다.

```bash
gh release download weights-260823 -R lcsvvo/Driver-Drowsiness-Detection -D model/artifacts
curl -L -o model/detectors/face_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
```

> **`face_landmarker.task` 를 빼먹기 쉽습니다.** 없어도 에러가 나지 않고 게이트만
> 조용히 꺼진 채 돌아갑니다. 그러면 입을 다물어도 YAWN 이 뜹니다. 아래 2번 셀이
> 그 경우를 잡아서 알려줍니다.


## 1. 환경 점검

In [ ]:
import sys, platform
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("실행파일:", sys.executable)

import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

print("\nTensorFlow :", tf.__version__)
print("OpenCV     :", cv2.__version__)
print("FaceDetectorYN (YuNet):", "OK" if hasattr(cv2, "FaceDetectorYN") else "없음 (OpenCV 4.5.4+ 필요)")

## 2. 경로 · 가중치 · 게이트

In [ ]:
import sys, json
from pathlib import Path

# =====================================================================
# 경로는 저장소 최상단 config.py 에서 가져온다. (README §10)
#
# 노트북에는 __file__ 이 없어서 config.py 의 "위치"만 cwd 기준으로 찾는다.
# 하지만 그 뒤의 모든 경로는 config.py 가 자신의 __file__ 로 계산하므로,
# 노트북을 어느 폴더에서 열든 결과가 같다. 못 찾으면 조용히 넘어가지 않고
# 즉시 멈춘다.
#
# src/ 도 sys.path 에 넣는다. 눈 전처리를 학습과 "같은 함수"
# (src/eye_preprocess.py) 로 통과시키기 위해서다.
# =====================================================================
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "config.py").exists()), None)
if _root is None:
    raise FileNotFoundError(
        f"config.py 를 찾지 못했습니다 (탐색 시작: {_here}).\n"
        "저장소를 clone 한 폴더 안에서 노트북을 열었는지 확인하세요."
    )
for _p in (str(_root), str(_root / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import config

PROJECT_ROOT = config.PROJECT_ROOT
ARTIFACT_DIR = config.ARTIFACT_DIR
YUNET_MODEL  = config.YUNET_MODEL

# =====================================================================
# 눈 모델 선택 — src/train_eye.py 의 A/B/C/C' 비교에서 가장 좋았던 것
#
#   A  mrl only            Closed-Recall 0.853 / acc 0.776
#   B  dmd only            Closed-Recall 0.597 / acc 0.951
#   C  mrl+dmd  gray128    Closed-Recall 0.914 / acc 0.962   <- 채택
#   C' mrl -> dmd 미세조정  Closed-Recall 0.881 / acc 0.963
#   (부록) C 를 256 RGB 로  Closed-Recall 0.441 / acc 0.931   <- 옛 eye_model 과 같은 규격
#
# 모두 DMD hold-out 을 "프레임 단위"로 집계한 test 값이다. 좌·우 눈은 같은 프레임에서
# 나오므로 crop 단위로 세면 독립 표본 수가 두 배로 과대 계산된다.
#
# 입력 규격(size/gray/sharpen)과 판정 임계값은 여기에 손으로 적지 않고 학습이 함께
# 남긴 _metrics.json 에서 읽는다. 모델 파일 이름만 바꾸면 규격도 따라온다.
# =====================================================================
EYE_MODEL_NAME = "eye_mrl+dmd__eval-dmd__gray128.keras"

EYE_MODEL_PATH   = ARTIFACT_DIR / EYE_MODEL_NAME
EYE_METRICS_PATH = EYE_MODEL_PATH.with_name(EYE_MODEL_PATH.stem + "_metrics.json")

# =====================================================================
# 하품 모델 선택 — yawn_mouthopen_v2 (DMD + YawDD 합본, 입 벌림 게이트 0.05)
#
# 데이터셋을 다시 만들었다. 두 출처 모두에서 입을 다문 프레임을 빼고, DMD 의
# yawn_without_hand + no_yawn 과 YawDD 의 yawning + normal/talking 을 합쳤다
# (scripts/build_yawn_mouthopen_dataset.py). 남은 문제는 "입은 벌어졌다,
# 하품인가 말하기인가" 하나다. 추론에도 같은 게이트를 단다 - 아래 YAWN_GATE.
#
# v2 test 1,553장 @0.50 (게이트를 통과한 프레임만 모은 셋. 말하기가 섞여 있어
# Precision 이 '말하는 사람을 하품으로 보는가'를 잰다):
#
#   zoo-cnn_large      acc 0.752  R 0.690  P 0.812  F1 0.746   <- 채택
#   distill-cnn_small  acc 0.697  R 0.603  P 0.773  F1 0.678
#   zoo-mbnetv2_035    acc 0.625  R 0.727  P 0.624  F1 0.671
#   zoo-cnn_small(KD X) acc 0.668 R 0.585  P 0.732  F1 0.650
#
# **distillation 을 했지만 teacher 를 채택했다.** student(cnn_small)가 teacher
# (cnn_large)보다 모든 지표에서 낮은데, cnn_small 은 Flatten 기반이라 파라미터가
# 오히려 더 많다(822,018 vs 622,402). 즉 이 조합에서는 압축 이득이 없다 -
# distillation 의 존재 이유가 성립하지 않는다. 속도도 14.6ms 대 7.1ms 로 둘 다
# 실시간에 충분하다. student 로 되돌리려면 아래 이름만 바꾸면 된다:
#   yawn_yawn_mouthopen_v2__distill-cnn_small__from-cnn_large+mbnetv2_035__T4-a0.3__eval-face__gray128.keras
#
# 엔드투엔드 (DMD test face 1,131장, 게이트 포함, 입 다문 프레임까지 전부.
# 프레임 단위 GT 라 이 숫자가 실제 동작에 가장 가깝다):
#
#                        acc     Recall   Precision
#   260821 릴리스(구)    0.654    0.243     0.984
#   v2 cnn_large        0.779    0.518     0.993   <- 채택
#
#   라벨별로 쪼개면:
#     손 안 가린 하품    Recall 0.878   (게이트 상한 0.895 에 거의 닿았다)
#     손으로 가린 하품   Recall 0.162   <- 여기가 남은 손실이다
#     오경보 (no_yawn)   0.003
#
# **손으로 가린 하품은 이 구조로는 거의 못 잡는다.** 랜드마커가 손 뒤의 입을
# "다물었다"고 재기 때문에 게이트가 79% 를 끊는다. 랜드마크가 실패하면 통과시키는
# fail-open 을 넣어 뒀지만, mediapipe 는 실패하지 않고 '닫힘'으로 잘못 재서 그
# 장치가 여기서는 거의 작동하지 않는다. DMD 하품 프레임의 절반이 손가림이라
# 전체 Recall 이 0.518 에서 멈추는 이유가 이것이다. 손-입 겹침을 따로 검출하는
# 경로가 필요하다 - 지금 구조 안에서 임계값을 만져서는 해결되지 않는다.
# =====================================================================
YAWN_MODEL_NAME = "yawn_yawn_mouthopen_v2__zoo-cnn_large__eval-face__gray128.keras"

YAWN_MODEL_PATH   = ARTIFACT_DIR / YAWN_MODEL_NAME
YAWN_METRICS_PATH = YAWN_MODEL_PATH.with_name(YAWN_MODEL_PATH.stem + "_metrics.json")

#: metrics.json 을 못 찾을 때만 쓰는 값. 위 모델의 실제 학습 설정과 같다.
DEFAULT_EYE_SPEC = {"size": 128, "gray": True, "sharpen": False, "threshold": 0.93}


def load_eye_spec(metrics_path):
    """학습이 남긴 metrics.json 에서 입력 규격과 판정 임계값을 읽는다.

    임계값은 test 가 아니라 val 에서 Closed-Recall 0.9 를 맞추도록 고른 값이다
    (src/train_eye.py 의 pick_threshold). test 에서 고르면 낙관 편향이 생긴다.
    """
    if not metrics_path.exists():
        print(f"[!] {metrics_path.name} 이 없어 기본 규격을 씁니다: {DEFAULT_EYE_SPEC}")
        return dict(DEFAULT_EYE_SPEC)

    m = json.loads(metrics_path.read_text(encoding="utf-8"))

    return {
        "size": int(m["size"]),
        "gray": bool(m["gray"]),
        "sharpen": bool(m["sharpen"]),
        "threshold": float(m["threshold"]),
    }


EYE_SPEC = load_eye_spec(EYE_METRICS_PATH)

# ---------------------------------------------------------------------
# 하품 입력 규격
#
# 새 모델은 '얼굴 crop' 을 받고, 옛 yawn_model.keras 는 '프레임 전체'를 받는다.
# 둘은 호환되지 않으므로 규격에 input 항목을 둬서 추론부가 갈라지게 한다.
# ---------------------------------------------------------------------
#: 판정 임계값. metrics.json 에도 threshold 가 있지만 **쓰지 않는다.**
#: 그 값은 val 에서 Yawn-Recall 0.90 을 맞추도록 고른 것인데, 하품 모델에서는
#: 역효과였다(A 조건은 0.15 까지 내려가 정확도가 0.503 으로 떨어진다).
#: 눈 모델에서 잘 통한 방식이 여기서는 통하지 않아 0.50 으로 고정한다.
#: 근거는 docs/yawn_model.md §3.
#: 새 distill-cnn_small 도 마찬가지다. metrics 의 0.22 로 판정하면 같은 test 에서
#: acc 0.747 -> 0.591, Precision 0.889 -> 0.584 로 떨어진다(Recall 은 0.607 -> 0.837).
YAWN_THRESHOLD = 0.50


def load_yawn_spec(metrics_path):
    """하품 모델의 입력 규격. .keras 와 짝이 되는 metrics.json 에서 읽는다."""
    if not metrics_path.exists():
        raise FileNotFoundError(
            f"{metrics_path.name} 이 없습니다. .keras 와 _metrics.json 은 항상 "
            "같이 다녀야 합니다 - 노트북이 이 파일에서 입력 규격을 읽습니다.")

    m = json.loads(metrics_path.read_text(encoding="utf-8"))

    return {
        "size": int(m["size"]),
        "gray": bool(m["gray"]),
        "sharpen": bool(m["sharpen"]),
        "threshold": YAWN_THRESHOLD,
        "threshold_in_metrics": float(m["threshold"]),   # 참고용. 판정에는 안 쓴다
    }


YAWN_SPEC = load_yawn_spec(YAWN_METRICS_PATH)

# ---------------------------------------------------------------------
# 입 벌림 게이트
#
# 하품 CNN 은 "입은 벌어졌다. 하품인가 말하기인가" 만 푼다. 입을 다물었는지는
# CNN 이 아니라 랜드마크로 먼저 가른다.
#
#     open_ratio <= threshold  ->  CNN 을 부르지 않고 p(yawn)=0
#     open_ratio >  threshold  ->  CNN 에 물어본다
#
# 학습 데이터(scripts/build_yawn_mouthopen_dataset.py)가 두 클래스 모두에서 입 다문
# 프레임을 빼고 만들어졌기 때문이다. 입 다문 프레임은 CNN 이 본 적 없는 입력이다.
#
# **임계값이 곧 Recall 상한이다.** 게이트에 걸린 하품은 모델이 아무리 좋아도 못 잡는다.
# DMD 하품 프레임(n=1,148, 프레임 단위 GT) 기준: 0.03 -> 0.922, 0.05 -> 0.895,
# 0.08 -> 0.856, 0.10 -> 0.817.
#
# 임계값은 src/mouth_gate.py 한 곳에서만 정의한다. 데이터셋 빌더도 같은 값을 쓴다.
# ---------------------------------------------------------------------
# src/ 의 모듈은 **매번 새로 읽는다.**
# Python 은 한 번 import 한 모듈을 sys.modules 에 캐시하므로, src/mouth_gate.py 를
# 고치고 이 셀을 다시 실행해도 옛 코드가 그대로 돈다. 커널을 재시작해야만 바뀌는데,
# 그걸 모르면 "고쳤는데 왜 그대로지" 로 한참 헤맨다.
import importlib
import mouth_gate
importlib.reload(mouth_gate)

LANDMARKER_PATH = PROJECT_ROOT / "model" / "detectors" / "face_landmarker.task"

YAWN_GATE = {
    "enabled": LANDMARKER_PATH.exists(),
    "threshold": mouth_gate.DEFAULT_THRESHOLD,
    "landmarker": LANDMARKER_PATH,
    # 프레임 단위 랜드마크는 흔들려서 임계값 근처에서 게이트가 깜빡인다.
    # 최근 몇 프레임의 최댓값을 쓰면 하품 시작을 놓치지 않으면서 깜빡임이 없어진다.
    "hold": 3,
}


def check_gate_matches_dataset(gate, model_path):
    """학습 데이터셋이 남긴 임계값과 게이트 임계값이 같은지 확인한다.

    다르면 조용히 어긋난다 - 게이트가 더 낮으면 학습에서 본 적 없는 구간이 CNN 에
    들어오고, 더 높으면 학습 데이터의 일부가 추론에서 영영 안 쓰인다. 데이터셋 폴더가
    없는 환경(추론만 하는 경우)에서는 확인을 건너뛴다.
    """
    # 모델 태그 앞부분이 데이터셋 폴더 이름이다 (train_yawn.py 의 tag_prefix 규칙).
    pkg = model_path.name[len("yawn_"):].split("__")[0]
    for base in (PROJECT_ROOT.parent / "Dataset", PROJECT_ROOT / "data" / "Dataset"):
        doc = base / pkg / "metadata" / "dataset.json"
        if not doc.exists():
            continue
        built = json.loads(doc.read_text(encoding="utf-8"))["mouth_open"]["threshold"]
        if abs(built - gate["threshold"]) > 1e-9:
            print(f"[!] 게이트 임계값이 학습 데이터셋과 다릅니다: "
                  f"추론 {gate['threshold']} vs 데이터셋 {built} ({pkg})")
            print("    src/mouth_gate.py 의 DEFAULT_THRESHOLD 를 맞추거나 "
                  "데이터셋을 그 값으로 다시 만드세요.")
        return built
    return None


GATE_BUILT_WITH = check_gate_matches_dataset(YAWN_GATE, YAWN_MODEL_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT.name)
print()

missing = []
for label, p in [("Eye CNN  ", EYE_MODEL_PATH),
                 ("Yawn CNN ", YAWN_MODEL_PATH),
                 ("YuNet    ", YUNET_MODEL)]:
    if p.exists():
        print(f"  [o] {label} {p.name:44s} {p.stat().st_size/1024/1024:7.2f} MB")
    else:
        print(f"  [X] {label} {p.name:44s} 없음")
        missing.append((label.strip(), p))

print()
print(f"  눈 입력 규격: {EYE_SPEC['size']}x{EYE_SPEC['size']} "
      f"{'gray' if EYE_SPEC['gray'] else 'rgb'}, sharpen={EYE_SPEC['sharpen']}")
print(f"  눈 판정 임계: p(Closed) >= {EYE_SPEC['threshold']:.2f}   (val 에서 선택)")
print(f"  하품 입력   : {YAWN_SPEC['size']}x{YAWN_SPEC['size']} "
      f"{'gray' if YAWN_SPEC['gray'] else 'rgb'} 얼굴 crop")
print(f"  하품 판정 임계: p(Yawn) >= {YAWN_SPEC['threshold']:.2f}")
if YAWN_GATE["enabled"]:
    print(f"  입 벌림 게이트: open_ratio > {YAWN_GATE['threshold']:.2f} 일 때만 CNN 호출"
          f"  (hold {YAWN_GATE['hold']} 프레임)")
    if GATE_BUILT_WITH is None:
        print("    학습 데이터셋 폴더가 없어 임계값 일치는 확인하지 못했습니다")
    elif abs(GATE_BUILT_WITH - YAWN_GATE["threshold"]) > 1e-9:
        print(f"    !! 학습 데이터셋은 {GATE_BUILT_WITH} 로 만들어졌습니다 (위 경고 참고)")
    else:
        print("    학습 데이터셋과 같은 임계값입니다")
else:
    print(f"  입 벌림 게이트: 꺼짐 - {LANDMARKER_PATH.name} 이 없습니다")
    print("    받는 곳: https://storage.googleapis.com/mediapipe-models/"
          "face_landmarker/face_landmarker/float16/1/face_landmarker.task")

if missing:
    print("\n" + "=" * 62)
    for label, p in missing:
        print(f"[!] {label} 가중치가 없습니다: {p.relative_to(PROJECT_ROOT)}")
    print("    Release 에서 받아 model/artifacts/ 에 넣으세요:")
    print("    https://github.com/lcsvvo/Driver-Drowsiness-Detection/releases/tag/weights-260823")
    print("=" * 62)
    raise FileNotFoundError("필요한 가중치가 없습니다.")

print("\n필요한 파일이 모두 준비됐습니다.")


## 3. Detector 정의

`detect_face()` 가 YuNet 출력을 dict 로 정리하고, 그 좌표로 눈과 얼굴을 자릅니다.

```python
face = {
    "box": [x, y, w, h],
    "confidence": score,
    "keypoints": {"left_eye":…, "right_eye":…, "nose":…,
                  "mouth_left":…, "mouth_right":…},
}
```

### 좌/우 이름이 서로 반대입니다

MTCNN 은 **이미지상 위치**로, YuNet 은 **인물 기준**으로 눈에 이름을 붙입니다. 같은
얼굴에서 두 검출기의 좌표를 비교하면 `left` 와 `right` 가 맞바뀌어 있습니다.
`_to_mtcnn_format()` 에서 뒤집어 맞춥니다.

### 눈과 하품이 다른 전처리를 씁니다

`extract_eyes` 가 돌려주는 것은 BGR crop 이 아니라 `preprocess_eye()` 를 통과한
**모델 입력 텐서**(128×128×1, 0~255)입니다. 학습(`src/mrl_dataset.py`)과 추론이 같은
함수를 쓰게 해서 입력 분포가 어긋나지 않게 합니다.

하품은 `yawn_infer_box()` 로 얼굴 높이의 **1.82배** 정사각형을 자릅니다. 학습 데이터가
그 비율로 만들어졌기 때문입니다. 여백식을 그대로 쓰면 얼굴이 학습 때보다 19% 크게
찍혀서 정확도가 0.630 → 0.485 로 떨어집니다(실측).

### 게이트는 여기와 실시간 루프 **양쪽에** 있습니다

`yawn_detection()` 에 하나, 아래 실시간 루프에 하나입니다. 루프는 속도 때문에
`yawn_detection()` 을 거치지 않고 모델을 직접 부르므로, 한쪽에만 넣으면 실시간에서만
게이트가 빠집니다.


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# 학습과 추론이 같은 전처리를 쓰게 하는 단일 출처. src/ 는 위 셀에서 sys.path 에 넣었다.
from eye_preprocess import preprocess_eye
from yawn_dataset import yawn_infer_box
from mouth_gate import MouthGate       # cell 4 에서 reload 한 모듈


# 입 벌림 게이트를 쓸지. 위 셀의 YAWN_GATE 가 규격을 들고 있다.
# 끄면(False) 옛 동작(모든 프레임을 CNN 에 넣기)으로 돌아간다.
YAWN_USE_GATE = True


class Drowsiness_Detector:
    """MTCNN 판과 동일한 인터페이스. 얼굴 검출만 YuNet 으로 교체."""

    def __init__(self, list_models, yunet_path=None, eye_spec=None, yawn_spec=None,
                 score_threshold=0.6, nms_threshold=0.3, top_k=5000,
                 gate_spec=None):

        self.list_models = [str(p) for p in list_models]

        # 눈 입력 규격 + 판정 임계값. 위 셀이 metrics.json 에서 읽어 넘긴다.
        # 학습과 다른 값을 쓰면 아무 에러 없이 성능만 조용히 떨어진다.
        self.eye_spec = dict(eye_spec if eye_spec is not None else EYE_SPEC)
        self.yawn_spec = dict(yawn_spec if yawn_spec is not None else YAWN_SPEC)

        # ---- YuNet 검출기 ----
        self.yunet_path = str(yunet_path or YUNET_MODEL)
        self.score_threshold = score_threshold

        self._input_size = (320, 240)      # setInputSize 로 계속 바뀐다
        self.detector = cv2.FaceDetectorYN.create(
            self.yunet_path, "", self._input_size,
            score_threshold, nms_threshold, top_k
        )

        # ---- 입 벌림 게이트 ----
        # 하품 모델은 "입은 벌어졌다, 하품인가 말하기인가"만 학습했다. 입을 다문
        # 프레임은 학습 데이터에 없으므로 CNN 에 넣으면 안 되고, 게이트에서 끊는다.
        # 규격(임계값)은 학습 데이터셋이 남긴 dataset.json 에서 온다 - 위 셀 참고.
        gate_spec = gate_spec if gate_spec is not None else globals().get("YAWN_GATE")
        self.gate_spec = dict(gate_spec) if gate_spec else None
        self.gate = None
        if YAWN_USE_GATE and self.gate_spec and self.gate_spec.get("enabled"):
            self.gate = MouthGate(self.gate_spec.get("landmarker"),
                                  threshold=self.gate_spec["threshold"],
                                  hold=self.gate_spec.get("hold", 3))

        #: 마지막 프레임의 open_ratio. 표시용이라 반환값 모양은 건드리지 않는다.
        #: 게이트를 못 쟀으면 None.
        self.last_open_ratio = None
        self.last_gate_open = True

        # Eye / Yawn 모델 로드
        self.models = self.get_models_ready()

    # =========================================================
    # 1. 모델 로드
    # =========================================================
    def get_models_ready(self):
        eye_model = tf.keras.models.load_model(self.list_models[0])
        yawn_model = tf.keras.models.load_model(self.list_models[1])
        return eye_model, yawn_model

    # =========================================================
    # 2. 이미지 Sharpen
    #    눈 전처리는 preprocess_eye 가 맡으므로 여기서 부르지 않는다.
    #    (현재 모델은 sharpen=False 로 학습됐다.) 외부 호환용으로만 남긴다.
    # =========================================================
    @staticmethod
    def sharpen(img):
        kernel = np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ])
        return cv2.filter2D(src=img, ddepth=-1, kernel=kernel)

    # =========================================================
    # 3. YuNet 출력 -> MTCNN 형식 dict
    # =========================================================
    @staticmethod
    def _to_mtcnn_format(f):
        """YuNet 의 15개 값을 MTCNN 과 같은 dict 로 변환.

        YuNet 원시 출력:
          [0:4]   x, y, w, h
          [4:6]   right_eye   (인물 기준 오른쪽 = 이미지 왼쪽)
          [6:8]   left_eye    (인물 기준 왼쪽  = 이미지 오른쪽)
          [8:10]  nose
          [10:12] mouth_right (인물 기준)
          [12:14] mouth_left  (인물 기준)
          [14]    score

        MTCNN 은 이미지상 위치로 이름을 붙이므로 좌/우를 서로 바꿔 매핑한다.
        """
        x, y, w, h = (int(v) for v in f[0:4])

        return {
            "box": [x, y, w, h],
            "confidence": float(f[14]),
            "keypoints": {
                # 교차 매핑: YuNet right_eye -> MTCNN left_eye
                "left_eye":    (int(f[4]),  int(f[5])),
                "right_eye":   (int(f[6]),  int(f[7])),
                "nose":        (int(f[8]),  int(f[9])),
                "mouth_left":  (int(f[10]), int(f[11])),
                "mouth_right": (int(f[12]), int(f[13])),
            },
        }

    # =========================================================
    # 4. 얼굴 검출 (YuNet)
    # =========================================================
    def detect_face(self, image):
        """가장 큰 얼굴 하나를 MTCNN 형식으로 반환. 없으면 None.

        YuNet 은 BGR 을 그대로 받으므로 MTCNN 판에 있던 cvtColor 가 필요 없다.
        """
        h, w = image.shape[:2]

        # 입력 크기가 바뀌면 알려줘야 한다
        if (w, h) != self._input_size:
            self.detector.setInputSize((w, h))
            self._input_size = (w, h)

        _, faces = self.detector.detect(image)

        if faces is None or len(faces) == 0:
            return None

        # 가장 큰 얼굴
        best = max(faces, key=lambda f: f[2] * f[3])

        return self._to_mtcnn_format(best)

    # =========================================================
    # 5. 양쪽 눈 영역 추출
    #    crop 위치는 MTCNN 판과 동일(얼굴 폭의 22%). DMD 학습 데이터를 만든
    #    src/build_dmd_eye_dataset.py 도 같은 22% 를 쓴다.
    #    달라진 것은 반환값 — 이제 '전처리가 끝난 모델 입력 텐서'다.
    # =========================================================
    def extract_eyes(self, image, face=None):

        if face is None:
            face = self.detect_face(image)

        if face is None:
            return None

        keypoints = face["keypoints"]

        left_eye = keypoints["left_eye"]
        right_eye = keypoints["right_eye"]

        h, w = image.shape[:2]

        face_width = face["box"][2]
        eye_size = max(int(face_width * 0.22), 20)

        def crop_eye(center):
            x, y = center

            x1 = max(int(x - eye_size), 0)
            y1 = max(int(y - eye_size), 0)
            x2 = min(int(x + eye_size), w)
            y2 = min(int(y + eye_size), h)

            eye = image[y1:y2, x1:x2]

            if eye.size == 0:
                return None, None

            # 학습과 '같은 함수'로 전처리한다. resize/gray/sharpen 을 여기서
            # 손으로 하면 학습 입력과 어긋난다.
            # 반환 shape: (size, size, 1) 또는 (size, size, 3), 값 범위 0~255.
            eye = preprocess_eye(eye,
                                 size=self.eye_spec["size"],
                                 grayscale=self.eye_spec["gray"],
                                 do_sharpen=self.eye_spec["sharpen"],
                                 input_is_bgr=True)

            return eye, (x1, y1, x2, y2)

        left, left_box = crop_eye(left_eye)
        right, right_box = crop_eye(right_eye)

        if left is None or right is None:
            return None

        return left, right, left_box, right_box, face

    # =========================================================
    # 7. Eye CNN 추론
    #    argmax(= 임계값 0.5) 가 아니라 val 에서 고른 임계값으로 판정한다.
    # =========================================================
    def eye_score(self, p_closed):
        """Closed 확률을 '0.5 가 판정선' 이 되도록 구간별 선형 변환한다.

        이 모델의 판정 임계값은 0.5 가 아니라 metrics.json 의 값(0.93)이다.
        val 에서 Closed-Recall 0.9 를 맞추도록 고른 값인데, 0.5 로 읽으면 같은
        test 에서 Closed-Precision 이 0.787 -> 0.638 로 떨어진다. 즉 뜬 눈을
        감았다고 하는 오경보가 크게 늘어난다.

        그런데 아래 EMA·PERCLOS·막대그래프는 모두 0.5 를 기준으로 쓴다. 원본
        확률을 그대로 넘기면 기준이 어긋나므로, 순서를 보존한 채 임계값만 0.5 로
        옮긴다. 0 -> 0, thr -> 0.5, 1 -> 1 을 잇는 직선 두 개다.
        """
        thr = self.eye_spec["threshold"]

        if p_closed < thr:
            return 0.5 * p_closed / max(thr, 1e-6)

        return 0.5 + 0.5 * (p_closed - thr) / max(1.0 - thr, 1e-6)

    def yawn_score(self, p_yawn):
        """하품 확률을 '0.5 가 판정선' 이 되도록 재조정한다. eye_score 와 같은 이유.

        지금은 임계값이 0.50 이라 이 함수가 항등변환이지만, 임계값을 바꿔도
        아래 EMA·막대그래프의 기준(0.5)이 따라오도록 통로를 만들어 둔다.
        """
        thr = self.yawn_spec["threshold"]

        if p_yawn < thr:
            return 0.5 * p_yawn / max(thr, 1e-6)

        return 0.5 + 0.5 * (p_yawn - thr) / max(1.0 - thr, 1e-6)

    def eye_detection(self, left, right):
        """전처리된 좌·우 눈 텐서 -> (감김 여부, 원시 예측, 보정 점수)."""

        # 색 변환·리사이즈는 extract_eyes 의 preprocess_eye 가 이미 끝냈다.
        # 모델 안에 Rescaling(1/255) 이 있으므로 0~255 그대로 입력한다.
        inputs = np.stack([left, right]).astype(np.float32)

        predictions = self.models[0].predict(inputs, verbose=0)

        # class 0 = Closed. 한쪽이라도 감기면 Closed 이므로 max
        p_closed = float(max(predictions[0][0], predictions[1][0]))
        score = self.eye_score(p_closed)

        return score >= 0.5, predictions, score

    # =========================================================
    # 7. Yawn CNN 추론  (게이트 -> CNN)
    # =========================================================
    def yawn_crop(self, image, face):
        """학습 데이터가 잘린 방식과 같은 정사각 얼굴 crop + 표시용 박스.

        학습 데이터는 세션 전체 검출의 합집합으로 만든 '고정 박스' 하나를 썼다.
        실시간에는 프레임이 하나뿐이라 그 여유가 안 생기므로, 같은 여백식을 쓰면
        얼굴이 학습 때보다 19% 크게 찍힌다. src/yawn_dataset.py 의 yawn_infer_box 는
        train/val 세션에서 잰 스케일(얼굴 높이의 1.82배)로 맞춰 자른다.
        """
        h, w = image.shape[:2]
        x1, y1, x2, y2 = yawn_infer_box(face["box"], w, h)

        crop = image[y1:y2, x1:x2]
        if crop.size == 0:
            return None, None

        return crop, (x1, y1, x2, y2)

    def yawn_detection(self, image, face=None):
        """하품 추론. 반환 (하품여부, 원시 예측, 표시용 박스)."""
        spec = self.yawn_spec

        if face is None:
            face = self.detect_face(image)
        if face is None:
            return False, None, None

        crop, box = self.yawn_crop(image, face)
        if crop is None:
            return False, None, None

        # ---- 입 벌림 게이트 ----
        # 학습 데이터는 두 클래스 모두 "입을 벌린" 프레임만 남기고 만들어졌다.
        # 입을 다문 프레임은 CNN 이 본 적 없는 입력이라 출력이 아무 의미가 없다.
        # 그래서 여기서 끊고 p(yawn)=0 으로 둔다. CNN 을 건너뛰므로 더 빠르기도 하다.
        #
        # 랜드마크를 못 잰 프레임은 통과시킨다(fail-open). 손으로 입을 가린 하품이
        # 그 경우인데, 거기서 닫아 버리면 그 자세를 통째로 잃는다.
        if self.gate is not None:
            gate_open, self.last_open_ratio = self.gate.check(crop)
            self.last_gate_open = gate_open
            if not gate_open:
                return False, np.array([0.0, 1.0], np.float32), box
        else:
            self.last_open_ratio, self.last_gate_open = None, True

        # 눈과 같은 전처리 함수를 통과시킨다.
        target = preprocess_eye(crop, size=spec["size"], grayscale=spec["gray"],
                                do_sharpen=spec["sharpen"], input_is_bgr=True)

        inputs = np.expand_dims(target, axis=0).astype(np.float32)
        prediction = self.models[1].predict(inputs, verbose=0)[0]

        # class 0 = yawn, class 1 = no_yawn.
        # argmax(= 임계값 0.5) 가 아니라 규격의 임계값으로 판정한다.
        yawn = float(prediction[0]) >= spec["threshold"]

        return yawn, prediction, box

print("Drowsiness_Detector (YuNet) 정의 완료")
print(f"  눈 입력 = {EYE_SPEC['size']}x{EYE_SPEC['size']}"
      f" {'gray' if EYE_SPEC['gray'] else 'rgb'}"
      f" / 판정 임계값 = {EYE_SPEC['threshold']:.2f}")
print(f"  하품 입력 = {YAWN_SPEC['size']}x{YAWN_SPEC['size']}"
      f" {'gray' if YAWN_SPEC['gray'] else 'rgb'}"
      f" (얼굴 crop)"
      f" / 판정 임계값 = {YAWN_SPEC['threshold']:.2f}")

## 4. Detector 로딩

In [ ]:
detector = Drowsiness_Detector(
    [EYE_MODEL_PATH, YAWN_MODEL_PATH],
    yunet_path=YUNET_MODEL,
    eye_spec=EYE_SPEC,          # metrics.json 에서 읽은 입력 규격·판정 임계값
    yawn_spec=YAWN_SPEC,
    gate_spec=YAWN_GATE,        # 입 벌림 게이트
)

print("YuNet + 두 CNN + 게이트 로딩 완료")

# 어떤 파일이 실제로 로드됐는지 확인한다. src/ 를 고쳤는데 반영이 안 될 때
# 여기부터 본다 (모듈 캐시 문제라면 경로는 같아도 내용이 옛것이다).
import mouth_gate as _mg
print(f"  게이트 코드 : {Path(_mg.__file__).name}  임계값 {_mg.DEFAULT_THRESHOLD}")


## 5. 카메라 점검

> **`cv2.imshow` 는 노트북에서 쓰지 않습니다.** VSCode 노트북 커널은 GUI 이벤트 루프가
> 없어 창이 멈추거나 키 입력을 못 받습니다. 아래는 프레임을 **노트북 안에 인라인으로**
> 갱신합니다.

백엔드×인덱스 조합을 시험하고 **읽기 속도까지 재서** 가장 빠른 것을 고릅니다. 어두운
곳에서 자동 노출이 셔터를 길게 잡으면 카메라가 10 FPS 까지 떨어지는데, 열리는지만
봐서는 그 경우를 못 잡기 때문입니다.


In [ ]:
import cv2, time
import numpy as np


def measure_camera(cap, n=25, warm=12):
    """cap.read() 평균 소요시간(초). 실패하면 None."""
    for _ in range(warm):
        cap.read()
    ts = []
    for _ in range(n):
        t = time.perf_counter()
        ok, f = cap.read()
        ts.append(time.perf_counter() - t)
        if not ok:
            return None
    return float(np.mean(ts))


def check_cameras(max_index=2):
    """백엔드 x 인덱스를 시험하고 '읽기 속도까지' 재서 가장 빠른 조합을 고른다.

    Windows 에서 DSHOW + 자동노출 조합은 어두운 환경에서 프레임레이트가
    1/3 로 떨어지는 일이 흔하다. 그래서 열리는지만 보지 않고 속도를 잰다.
    """
    backends = [("CAP_MSMF", cv2.CAP_MSMF),
                ("CAP_DSHOW", cv2.CAP_DSHOW),
                ("CAP_ANY", cv2.CAP_ANY)]
    working = []

    for bname, bid in backends:
        for idx in range(max_index):
            cap = None
            try:
                cap = cv2.VideoCapture(idx, bid)
                if not cap.isOpened():
                    continue
                dt = measure_camera(cap)
                if dt is None:
                    continue
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                print(f"  {bname:10s} index={idx}  {w}x{h}  "
                      f"{dt*1000:6.1f} ms  ({1/dt:4.1f} FPS)")
                working.append((dt, bname, bid, idx))
            except Exception as e:
                print(f"  {bname:10s} index={idx}  오류 {e}")
            finally:
                if cap is not None:
                    cap.release()

    if not working:
        print("사용 가능한 카메라가 없습니다.")
        print("  - 다른 앱(Zoom/Teams/카메라앱)이 점유 중인지 확인")
        print("  - Windows 설정 > 개인 정보 및 보안 > 카메라 접근 허용")
        return []

    working.sort()          # 가장 빠른 것이 앞으로
    return working


print("카메라 탐색 중... (백엔드별 읽기 속도까지 측정)\n")
CAMERAS = check_cameras()

if CAMERAS:
    best_dt, best_name, CAMERA_BACKEND, CAMERA_INDEX = CAMERAS[0]
    print(f"\n선택: {best_name}  index={CAMERA_INDEX}  "
          f"({1/best_dt:.1f} FPS)")

    if best_dt > 0.05:
        print("\n[!] 읽기가 느립니다 (>50ms). 자동 노출이 원인일 수 있습니다.")
        print("    _open_camera(manual_exposure=True) 로 열어보세요.")
else:
    CAMERA_INDEX, CAMERA_BACKEND = 0, cv2.CAP_ANY

In [ ]:
import os, time, threading
import cv2
from pathlib import Path
from IPython.display import display, Image


def _open_camera(camera_index=None, backend=None,
                 manual_exposure=False, exposure=-6):
    """카메라를 열어 VideoCapture 반환. 실패 시 예외.

    manual_exposure=True 면 자동 노출을 끈다. 어두운 곳에서 카메라가
    셔터를 길게 잡아 프레임레이트가 떨어지는 것을 막지만 화면은 어두워진다.
    """
    idx = CAMERA_INDEX if camera_index is None else camera_index
    bid = CAMERA_BACKEND if backend is None else backend

    cap = cv2.VideoCapture(idx, bid)

    if not cap.isOpened():
        cap.release()
        raise RuntimeError(
            f"웹캠을 열 수 없습니다 (index={idx}).\n"
            "  - 위 check_cameras() 결과를 확인하세요\n"
            "  - 다른 앱이 카메라를 점유 중인지 확인\n"
            "  - Windows 설정 > 개인 정보 및 보안 > 카메라 접근 허용"
        )

    if manual_exposure:
        cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)   # 0.25=수동, 0.75=자동
        cap.set(cv2.CAP_PROP_EXPOSURE, exposure)

    return cap


class FrameGrabber:
    """카메라를 백그라운드 스레드에서 계속 읽어 '가장 최근 프레임'만 유지한다.

    cap.read() 는 다음 프레임이 나올 때까지 블로킹한다. 메인 루프에서 직접
    호출하면 [카메라 대기 -> 추론] 이 직렬로 쌓이지만, 스레드로 분리하면
    추론하는 동안 카메라가 병렬로 프레임을 받아둬서 대기가 사라진다.
    """

    def __init__(self, cap):
        self.cap = cap
        self.lock = threading.Lock()
        self.frame = None
        self.running = True
        self.count = 0
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def _loop(self):
        while self.running:
            ok, f = self.cap.read()
            if not ok:
                self.running = False
                break
            with self.lock:
                self.frame = f
                self.count += 1

    def read(self, timeout=5.0):
        """가장 최근 프레임을 반환. 아직 없으면 잠깐 기다린다."""
        t0 = time.time()
        while True:
            with self.lock:
                if self.frame is not None:
                    return True, self.frame.copy()
            if not self.running or (time.time() - t0) > timeout:
                return False, None
            time.sleep(0.002)

    def read_new(self, last_seq, timeout=5.0):
        """직전에 처리한 것보다 '새로운' 프레임이 나올 때까지 기다렸다 반환.

        카메라는 30 FPS 인데 루프가 100 FPS 로 돌면 같은 프레임을 세 번씩
        추론하게 된다. 결과는 당연히 같으므로 순수 낭비다. 새 프레임에만
        연산을 쓰도록 시퀀스 번호로 거른다.
        """
        t0 = time.time()
        while True:
            with self.lock:
                if self.frame is not None and self.count != last_seq:
                    return True, self.frame.copy(), self.count
            if not self.running or (time.time() - t0) > timeout:
                return False, None, last_seq
            time.sleep(0.001)

    def stop(self):
        self.running = False
        self.thread.join(timeout=1.0)
        self.cap.release()


def _show_inline(frame, handle=None, quality=80):
    """BGR 프레임을 노트북에 인라인 표시. display handle 을 반환."""
    ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        return handle

    img = Image(data=buf.tobytes())

    if handle is None:
        return display(img, display_id=True)

    handle.update(img)
    return handle


## 6. 실시간 모니터  *(직접 실행)*

**정지: 셀 왼쪽의 ■ (interrupt) 버튼** 또는 `max_seconds` 만료.

| 막대 | 의미 |
|---|---|
| `EYE CLOSED` | 양쪽 눈 중 감김 확률이 높은 쪽. **0.5 이상이면 감김** |
| `YAWN` | 이 프레임의 하품 확률. 게이트가 닫히면 0 |
| `YAWN ACC` | **시간 누적된** 하품 점수. 0.5 이상이면 하품 판정 |
| `DROWSY` | 세 신호를 합친 **최종 판정**. 0.6 이상이면 경보 |
| `PERCLOS` | 최근 1분 중 눈이 감겨 있던 시간의 비율 |

### DROWSY 는 세 신호를 어떻게 합치나

```
eye_family = max(EYE 의 EMA, PERCLOS 정규화)          <- 같은 신호원이라 max
DROWSY     = eye_family + (1 - eye_family) * 0.6 * YAWN ACC
```

**EYE 와 PERCLOS 는 같은 신호원입니다.** 둘 다 눈 감김을 보고 시간 축만 다릅니다
(EYE 는 최근 몇 초, PERCLOS 는 최근 1분). 독립 증거처럼 더하면 눈 하나로 두 번
가산되어 점수가 부풀려집니다. 그래서 max 로 묶습니다.

하품은 다른 신호원이라 가산합니다. 가중치 0.6 은 **최대치 하품 하나만으로 0.60** —
기본 임계값에 겨우 닿는 값입니다. 확실한 하품은 혼자서도 경보를 내지만 어중간한
하품(0.8 -> 0.53)은 못 넘습니다. 하품 Precision 이 완벽하지 않으므로(말하기 오경보
0.195) 혼자서 쉽게 울리지 않게 했습니다.

PERCLOS 는 `0.15` 에서 0.5 가 되도록 정규화하고, **창이 덜 찼으면 결합에서 뺍니다** —
덜 찬 값을 그대로 쓰면 초반 1분이 과소평가됩니다.

실측 동작:

| 상황 | EYE | PERCLOS | YAWN | DROWSY | |
|---|---|---|---|---|---|
| 정상 주행 | 0.10 | 0.02 | 0.00 | 0.100 | |
| 하품 한 번 (눈은 멀쩡) | 0.10 | 0.02 | 1.00 | 0.640 | 경보 |
| 어중간한 하품만 | 0.10 | 0.02 | 0.80 | 0.532 | |
| 눈을 자주 감음 | 0.20 | 0.18 | 0.00 | 0.600 | 경보 |
| 미세수면 | 0.85 | 0.05 | 0.00 | 0.850 | 경보 |
| 눈 약간 + 하품 | 0.40 | 0.10 | 0.80 | 0.688 | 경보 |

`MOUTH` 줄에 게이트 상태가 실시간으로 뜹니다. **YAWN 이 왜 그 값인지** 여기서 바로
보입니다.

```
MOUTH 0.007 <= 0.05  CLOSED -> YAWN 0      입을 다물어 CNN 을 건너뜀
MOUTH 0.369 >  0.05  OPEN -> CNN           CNN 이 판정
```

### 하품 누적이 하는 일

프레임마다 네 줄입니다 (`src/yawn_accumulator.py`).

```python
dt   = min(t - 이전_t, 1.0)        # 프레임이 아니라 '초' 를 센다
step = (p_yawn - 0.5) * dt         # 판정선 위면 +, 아래면 −
if p_yawn < 0.5:  step *= 3.0      # 증거가 없으면 3배로 빨리 깎는다
acc  = clip(acc + step, 0.0, 1.0)
```

`acc` 의 단위는 **초**입니다. "확신을 갖고 몇 초를 버텼나" 를 재고,
`acc >= 0.15` 이면서 최근 3초 최대 `open_ratio >= 0.20` 일 때 하품으로 판정합니다.

25Hz 에서 확신 0.95 로 하품하면 **0.36초 만에 발화**하고, 입을 다물면 **0.16초 만에
해제**됩니다. 빨리 차고 더 빨리 식습니다 — 말하기는 CNN 이 튀어도 연속 0.33초를
못 버팁니다.

**프레임이 아니라 시간을 세는 이유**: 이 루프는 입을 벌리면 22Hz, 다물면 30Hz 로
돌고 PC 마다도 다릅니다. 프레임을 세면 빠른 PC 에서 더 빨리 발화합니다.

### 프레임 예산 (실측, CPU)

| | 입 벌림 | 입 다뭄 |
|---|---|---|
| 1회 반복 | 45.4 ms | 25.6 ms |
| 실효 속도 | 22 Hz | 30 Hz (카메라 상한) |

게이트가 닫히면 하품 CNN 을 건너뛰므로 그만큼 빨라집니다.


### PERCLOS — 최근 1분 중 눈이 감긴 시간 비율

`EYE CLOSED` 는 그 순간의 확률이고, `DROWSY` 는 EMA(지수이동평균)라 최근 몇 초에 강하게
쏠립니다. 둘 다 "지난 1분 동안 얼마나 감고 있었나"는 답하지 못합니다. 그걸 재는 게 PERCLOS 입니다.

**프레임 수가 아니라 시간으로 셉니다.**
FPS 가 27~29 사이에서 흔들리고 NO FACE 구간에서는 더 떨어지므로, 프레임을 세면
느린 구간이 과소평가됩니다. 프레임 간 간격(`dt`)을 더합니다.

**얼굴을 못 잡은 시간은 분모에서 뺍니다.**
안 그러면 고개를 돌린 시간이 통째로 "눈 뜬 시간"으로 들어가 PERCLOS 가 낮게 나옵니다.
대신 `coverage`(창 안에서 얼굴이 잡힌 시간 비율)를 같이 보고, 너무 낮으면 값을 신뢰하지 않습니다.

```
PERCLOS  = (눈 감긴 시간) / (얼굴이 잡힌 시간)     <- 최근 window_sec 구간
coverage = (얼굴이 잡힌 시간) / (창 전체 시간)
```

| 기준 | 값 |
|---|---|
| 각성 상태 | 보통 0.05 미만 |
| 졸음 경고 | 0.15 이상 (`perclos_threshold` 기본값) |

> 창이 1분이면 **1분이 지나야** 값이 제대로 찹니다. `max_seconds=60` 으로 돌리면
> 마지막 순간에야 창이 다 차므로, PERCLOS 를 보려면 `max_seconds=180` 정도로 길게 도세요.
> 창이 `min_window_sec`(기본 10초)만큼 차기 전에는 `warming up` 으로 표시합니다.

In [ ]:
import time
from collections import deque


class PerclosTracker:
    """최근 window_sec 동안 '눈이 감긴 시간 비율'(PERCLOS)을 계산한다.

    프레임을 세지 않고 프레임 간 간격(dt)을 더한다. FPS 가 일정하지 않기 때문이다.
    얼굴을 못 잡은 구간은 분모에서 빼고, 대신 coverage 로 신뢰도를 따로 보고한다.
    """

    def __init__(self, window_sec=60.0, closed_threshold=0.5,
                 min_window_sec=10.0, max_gap=0.5):
        self.window_sec = window_sec
        self.closed_threshold = closed_threshold
        self.min_window_sec = min_window_sec

        # 셀을 멈췄다 재개하거나 카메라가 스톨하면 dt 가 수 초로 튄다.
        # 그 한 프레임이 창을 통째로 채우지 않도록 자른다.
        self.max_gap = max_gap

        self.samples = deque()        # (t, dt, closed, face_ok)
        self.prev_t = None

        # 창과 무관한 세션 전체 누적
        self.total_time = 0.0
        self.face_time = 0.0
        self.closed_time = 0.0

        # 연속으로 감고 있던 최장 시간 (microsleep 탐지용)
        self.cur_closure = 0.0
        self.max_closure = 0.0

    def update(self, eye_p, face_ok, t=None):
        """한 프레임 반영. eye_p 는 '감김' 확률(class 0)."""
        t = time.time() if t is None else t

        if self.prev_t is None:       # 첫 프레임은 dt 를 모른다
            self.prev_t = t
            return

        dt = min(t - self.prev_t, self.max_gap)
        self.prev_t = t

        closed = bool(face_ok and eye_p >= self.closed_threshold)

        self.samples.append((t, dt, closed, face_ok))
        self.total_time += dt

        if face_ok:
            self.face_time += dt
            if closed:
                self.closed_time += dt
                self.cur_closure += dt
                self.max_closure = max(self.max_closure, self.cur_closure)
            else:
                self.cur_closure = 0.0
        else:
            self.cur_closure = 0.0    # 얼굴을 놓친 구간은 연속으로 잇지 않는다

        cutoff = t - self.window_sec
        while self.samples and self.samples[0][0] < cutoff:
            self.samples.popleft()

    def window(self):
        """최근 창 기준 (perclos, coverage, 창에 쌓인 시간, 신뢰 가능 여부)."""
        span = closed_t = face_t = 0.0
        for _, dt, closed, face_ok in self.samples:
            span += dt
            if face_ok:
                face_t += dt
                if closed:
                    closed_t += dt

        perclos = closed_t / face_t if face_t > 0 else 0.0
        coverage = face_t / span if span > 0 else 0.0
        ready = span >= self.min_window_sec and coverage >= 0.3

        return perclos, coverage, span, ready

    def session(self):
        """세션 전체 기준 (perclos, coverage, 총 시간, 최장 연속 감김 시간)."""
        perclos = self.closed_time / self.face_time if self.face_time > 0 else 0.0
        coverage = self.face_time / self.total_time if self.total_time > 0 else 0.0
        return perclos, coverage, self.total_time, self.max_closure


print("PerclosTracker 정의 완료")

In [ ]:
import numpy as np
from collections import deque

# cell 4 와 같은 이유로 새로 읽는다. 이 셀만 다시 실행해도 반영되게 여기에 둔다.
import importlib
import yawn_accumulator
importlib.reload(yawn_accumulator)
from yawn_accumulator import YawnAccumulator


def _draw_bar(img, label, value, y, color, width=260, height=18, x=15):
    cv2.rectangle(img, (x, y), (x + width, y + height), (60, 60, 60), -1)
    filled = int(width * float(np.clip(value, 0, 1)))
    cv2.rectangle(img, (x, y), (x + filled, y + height), color, -1)
    cv2.rectangle(img, (x, y), (x + width, y + height), (200, 200, 200), 1)
    cv2.putText(img, f"{label:11s} {value:0.2f}", (x + width + 10, y + height - 3),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)


def _draw_sparkline(img, hist, x, y, w, h, threshold):
    cv2.rectangle(img, (x, y), (x + w, y + h), (35, 35, 35), -1)
    cv2.rectangle(img, (x, y), (x + w, y + h), (120, 120, 120), 1)

    ty = int(y + h * (1 - threshold))
    for sx in range(x, x + w, 12):
        cv2.line(img, (sx, ty), (min(sx + 6, x + w), ty), (0, 0, 255), 1)

    if len(hist) >= 2:
        pts = []
        for i, v in enumerate(hist):
            px = x + int(w * i / (len(hist) - 1))
            py = y + int(h * (1 - float(np.clip(v, 0, 1))))
            pts.append((px, py))
        cv2.polylines(img, [np.array(pts, np.int32)], False, (0, 255, 255), 2)

    cv2.putText(img, "DROWSY history", (x + 5, y - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)


def combine_drowsy(eye_ema, perclos_n, yawn, yawn_weight=0.6):
    """세 신호 -> 하나의 졸음 점수. 각 입력은 '0.5 가 그 신호의 판정선' 규칙이다.

    **EYE 와 PERCLOS 는 같은 신호원이다.** 둘 다 눈 감김을 보고, 시간 축만 다르다
    (EYE 는 최근 몇 초의 EMA, PERCLOS 는 최근 1분 누적). 독립 증거처럼 서로 더하면
    눈 하나로 두 번 가산되어 점수가 부풀려진다. 그래서 둘은 max 로 묶는다.

    하품은 다른 신호원이라 가산한다. 노이즈-OR 꼴로 올리되 가중치를 둔다.

        drowsy = eye_family + (1 - eye_family) * w * yawn

    w=0.6 이면 **최대치 하품 하나만으로 0.60** 이다. 기본 임계값(0.6)에 겨우 닿으므로
    확실한 하품은 혼자서도 경보를 내지만, 어중간한 하품(0.8)은 0.48 로 못 넘는다 -
    눈 쪽 근거가 조금이라도 있어야 한다. 하품 Precision 이 완벽하지 않으니
    (말하기 오경보 0.195) 혼자서 쉽게 울리지 않게 하는 편이 낫다.

    perclos_n 이 None 이면(창이 덜 찼거나 얼굴을 너무 놓쳤으면) 빼고 계산한다.
    """
    eye_family = eye_ema if perclos_n is None else max(eye_ema, perclos_n)
    return min(1.0, eye_family + (1.0 - eye_family) * yawn_weight * yawn)


def run_realtime_scores(camera_index=None,
                        max_seconds=60,
                        detect_width=None,
                        ema_alpha=0.3,
                        threshold=0.6,
                        history_len=120,
                        mirror=True,
                        threaded=True,
                        manual_exposure=False,
                        perclos_window=60.0,
                        perclos_threshold=0.15,
                        eye_closed_threshold=0.5):
    """실시간 졸음 점수 모니터. 정지하려면 ■(interrupt) 버튼을 누르세요.

    threaded=True 면 카메라 읽기를 백그라운드 스레드로 분리해
    [카메라 대기 -> 추론] 직렬 구조를 없앤다.

    PERCLOS 는 최근 perclos_window 초 중 눈이 감겨 있던 시간의 비율이다.
    창이 다 차야 의미가 있으므로 max_seconds 를 창보다 넉넉히 잡으세요.

    eye_closed_threshold 는 모델의 원시 확률이 아니라 detector.eye_score() 로
    보정한 점수에 적용된다. 보정이 모델 임계값을 0.5 로 옮겨 놓으므로 기본값
    0.5 를 그대로 두면 된다.

    반환: 세션 요약 dict.
    """

    cap = _open_camera(camera_index, manual_exposure=manual_exposure)
    grabber = FrameGrabber(cap) if threaded else None

    last_seq = -1

    def get_frame():
        """새 프레임만 가져온다 (스레드 모드). 중복 추론 방지."""
        nonlocal last_seq
        if grabber:
            ok, f, seq = grabber.read_new(last_seq)
            last_seq = seq
            return ok, f
        return cap.read()

    handle = None
    hist = deque(maxlen=history_len)
    eye_ema = 0.0        # 눈 감김의 짧은 시간축
    drowsy_score = 0.0   # 세 신호를 합친 최종 점수

    perclos_tracker = PerclosTracker(window_sec=perclos_window,
                                     closed_threshold=eye_closed_threshold)
    perclos = coverage = span = 0.0
    perclos_ready = False

    eye_p = yawn_p = 0.0

    # 하품 확률을 시간축으로 누적한다. 프레임 한 장으로는 하품과 말하기를 가를 수
    # 없어서, "얼마나 오래 확신했나" + "얼마나 크게 벌렸나" 를 함께 본다.
    # 근거와 실측은 src/yawn_accumulator.py 참고.
    yawn_acc = YawnAccumulator()
    yawn_t = 0.0          # 표시용 0~1 연속값 ("얼마나 하품 같은가")
    yawn_alarm = 0.0      # DROWSY EMA 용 (0.5 가 판정선)
    boxes = []
    face_ok = False

    t0 = time.time()
    frames = 0
    fps = 0.0

    eye_net, yawn_net = detector.models

    try:
        while True:
            if max_seconds and (time.time() - t0) > max_seconds:
                break

            ok, frame = get_frame()
            if not ok:
                break

            if mirror:
                frame = cv2.flip(frame, 1)

            frames += 1
            h0, w0 = frame.shape[:2]

            # ---- 얼굴 검출 (YuNet) ----
            if detect_width and detect_width < w0:
                scale = detect_width / w0
                small = cv2.resize(frame, (detect_width, int(h0 * scale)))
                f = detector.detect_face(small)

                if f is not None:
                    inv = 1.0 / scale
                    face = {
                        "box": [int(v * inv) for v in f["box"]],
                        "confidence": f["confidence"],
                        "keypoints": {k: (int(v[0] * inv), int(v[1] * inv))
                                      for k, v in f["keypoints"].items()},
                    }
                else:
                    face = None
            else:
                face = detector.detect_face(frame)

            now = time.time()

            if face is not None:
                eye_result = detector.extract_eyes(frame, face=face)
                face_ok = eye_result is not None
            else:
                face_ok = False

            if face_ok:
                left, right, lbox, rbox, face = eye_result

                # left/right 는 extract_eyes 가 preprocess_eye 로 전처리를 끝낸 텐서다.
                # model(x) 직접 호출이 .predict() 보다 3배 빠르다
                inp = np.stack([left, right]).astype(np.float32)
                pe = eye_net(inp, training=False).numpy()

                # class 0 = Closed. 한쪽이라도 감기면 Closed 이므로 max.
                # 판정선이 0.5 로 오도록 보정해 아래 EMA·PERCLOS 와 기준을 맞춘다.
                eye_p = detector.eye_score(float(max(pe[0][0], pe[1][0])))

                # 하품: 입력 방식이 모델에 따라 다르다 (detector.yawn_spec["input"]).
                yspec = detector.yawn_spec
                mouth_box = None
                target = None

                crop, mouth_box = detector.yawn_crop(frame, face)

                # ---- 입 벌림 게이트 ----
                # detector.yawn_detection() 과 같은 판단을 여기서도 한다. 이 루프는
                # 속도 때문에 yawn_detection 을 거치지 않고 모델을 직접 부르므로,
                # 게이트를 여기에도 두지 않으면 실시간에서만 게이트가 빠진다.
                if crop is not None and detector.gate is not None:
                    gate_open, open_ratio = detector.gate.check(crop)
                    detector.last_open_ratio = open_ratio
                    detector.last_gate_open = gate_open
                    if not gate_open:
                        crop = None          # 입을 다물었다. CNN 을 부르지 않는다
                        yawn_p = 0.0

                if crop is not None:
                    target = preprocess_eye(crop, size=yspec["size"],
                                            grayscale=yspec["gray"],
                                            do_sharpen=yspec["sharpen"],
                                            input_is_bgr=True)

                if target is not None:
                    inp_y = np.expand_dims(target, 0).astype(np.float32)
                    py = yawn_net(inp_y, training=False).numpy()[0]

                    # class 0 = yawn. 눈과 마찬가지로 판정선이 0.5 로 오도록 보정한다.
                    yawn_p = detector.yawn_score(float(py[0]))

                # ---- 시간 누적 ----
                # 게이트에 걸려 CNN 을 건너뛴 프레임(yawn_p = 0)도 넣어야 한다.
                # 그래야 입을 다문 시간만큼 누적이 식는다.
                _a = yawn_acc.update(now, yawn_p, detector.last_open_ratio)
                yawn_t, yawn_alarm = _a["score"], _a["alarm"]

                boxes = [(lbox, (0, 255, 0)), (rbox, (0, 255, 0))]
                if mouth_box is not None:
                    boxes.append((mouth_box, (0, 0, 255)))
                x, y, fw, fh = face["box"]
                boxes.append(((x, y, x + fw, y + fh), (255, 0, 0)))
            else:
                eye_p *= 0.7
                yawn_p *= 0.7
                boxes = []
                # 얼굴을 놓친 구간은 연속 프레임이 아니다. hold 창에 남은 값이
                # 얼굴을 다시 잡은 순간의 게이트를 잘못 열지 않게 비운다.
                if detector.gate is not None:
                    detector.gate.reset()
                    detector.last_open_ratio = None
                # 누적은 리셋하지 않고 '증거 없음' 으로 갱신한다. 고개를 잠깐 돌린
                # 것만으로 쌓인 하품 증거가 사라지면 안 되고, 계속 못 보면 식어야 한다.
                _a = yawn_acc.update(now, 0.0, None)
                yawn_t, yawn_alarm = _a["score"], _a["alarm"]

            # ---- 시간축 누적 ----
            # EMA: 최근 몇 초에 반응하는 순간 지표
            # DROWSY 에는 **누적값** 을 쓴다. 순간 yawn_p 를 쓰면 말하다가 한 프레임
            # 튄 것만으로 경보가 오른다 - 지금 고치려는 그 증상이다.
            # EYE 는 짧은 시간축으로 눌러 준다 (깜빡임 한 번에 경보가 뜨지 않게).
            # 하품은 이미 YawnAccumulator 가 시간 누적을 했으므로 EMA 를 또 걸지
            # 않는다 - 두 번 누르면 반응이 굼떠진다.
            eye_ema = ema_alpha * eye_p + (1 - ema_alpha) * eye_ema

            # PERCLOS: 최근 perclos_window 초 중 눈이 감긴 시간 비율
            # (분모는 '얼굴이 잡힌 시간'. 고개 돌린 시간을 눈 뜬 시간으로 세지 않는다)
            perclos_tracker.update(eye_p, face_ok)
            perclos, coverage, span, perclos_ready = perclos_tracker.window()

            # PERCLOS 를 '0.5 가 판정선' 규칙으로 맞춘다. 창이 덜 찼으면 None 으로
            # 두어 결합에서 뺀다 - 덜 찬 값을 그대로 쓰면 초반에 과소평가된다.
            perclos_n = (min(0.5 * perclos / max(perclos_threshold, 1e-6), 1.0)
                         if perclos_ready else None)

            drowsy_score = combine_drowsy(eye_ema, perclos_n, yawn_alarm)
            hist.append(drowsy_score)

            drowsy = drowsy_score >= threshold
            perclos_alert = perclos_ready and perclos >= perclos_threshold

            # ---- 그리기 ----
            vis = frame.copy()
            for (x1, y1, x2, y2), c in boxes:
                cv2.rectangle(vis, (x1, y1), (x2, y2), c, 2)

            panel_h = 240
            overlay = vis.copy()
            cv2.rectangle(overlay, (0, h0 - panel_h), (w0, h0), (0, 0, 0), -1)
            vis = cv2.addWeighted(overlay, 0.55, vis, 0.45, 0)

            base = h0 - panel_h + 15
            _draw_bar(vis, "EYE CLOSED", eye_p, base, (0, 200, 255))
            _draw_bar(vis, "MOUTH OPEN", yawn_p, base + 26, (0, 140, 255))
            # YAWN ACC 는 0~1 연속값이다. 0.5 가 판정선이 아니므로(발화는 15%
            # 지점) 색으로 구분한다.
            _draw_bar(vis, "YAWN ACC", yawn_t, base + 52,
                      (0, 0, 255) if yawn_acc._on else (0, 140, 200))
            _draw_bar(vis, "DROWSY", drowsy_score, base + 78,
                      (0, 0, 255) if drowsy else (0, 220, 0))
            _draw_bar(vis, "PERCLOS", perclos, base + 104,
                      (0, 0, 255) if perclos_alert else
                      ((0, 220, 0) if perclos_ready else (120, 120, 120)))

            # 창이 차기 전과 얼굴을 자주 놓친 구간에서는 PERCLOS 를 믿을 수 없다
            if perclos_ready:
                note = f"win {span:0.0f}s  face {coverage*100:0.0f}%"
            elif span < perclos_tracker.min_window_sec:
                note = f"warming up {span:0.0f}/{perclos_tracker.min_window_sec:0.0f}s"
            else:
                note = f"low coverage {coverage*100:0.0f}%"
            cv2.putText(vis, note, (15, base + 137),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)

            # 입 벌림 게이트 상태. YAWN 이 왜 0 인지(혹은 왜 떴는지)를 눈으로 본다.
            if detector.gate is not None and face_ok:
                r = detector.last_open_ratio
                thr = detector.gate.threshold
                if r is None:
                    gtxt, gcol = "MOUTH   ?  (못 잼 -> CNN 에 넘김)", (170, 170, 170)
                elif detector.last_gate_open:
                    gtxt, gcol = f"MOUTH {r:0.3f} > {thr:0.2f}  OPEN -> CNN", (0, 220, 0)
                else:
                    gtxt, gcol = f"MOUTH {r:0.3f} <= {thr:0.2f}  CLOSED -> YAWN 0", (255, 170, 0)
                cv2.putText(vis, gtxt, (300, base + 137),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, gcol, 1)

            _draw_sparkline(vis, hist, 15, base + 156, 260, 52, threshold)

            if frames % 5 == 0:
                fps = frames / max(time.time() - t0, 1e-6)

            status = "DROWSY!" if drowsy else ("NORMAL" if face_ok else "NO FACE")
            scolor = (0, 0, 255) if drowsy else ((0, 220, 0) if face_ok else (150, 150, 150))
            cv2.putText(vis, status, (20, 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.1, scolor, 3)
            cv2.putText(vis, f"{fps:.1f} FPS   {time.time()-t0:.0f}s",
                        (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220, 220, 220), 1)

            if perclos_alert:
                cv2.putText(vis, f"PERCLOS {perclos*100:.0f}%", (20, 105),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            handle = _show_inline(vis, handle, quality=75)

    except KeyboardInterrupt:
        print("사용자 중지")

    finally:
        if grabber:
            grabber.stop()
        else:
            cap.release()

    s_perclos, s_coverage, s_total, s_max_closure = perclos_tracker.session()

    print(f"종료 — {frames} 프레임 처리, 평균 {fps:.1f} FPS")
    print()
    print(f"세션 {s_total:.0f}s  (얼굴 검출 {s_coverage*100:.0f}%)")
    print(f"  PERCLOS       : {s_perclos*100:5.1f} %   (임계 {perclos_threshold*100:.0f}%)")
    print(f"  최장 연속 감김: {s_max_closure:5.2f} s")

    if s_coverage < 0.3:
        print("  [!] 얼굴이 잡힌 시간이 너무 짧아 PERCLOS 를 신뢰하기 어렵습니다.")
    elif s_perclos >= perclos_threshold:
        print("  [!] PERCLOS 가 임계를 넘었습니다 — 졸음 의심")

    return {
        "frames": frames,
        "fps": fps,
        "perclos": s_perclos,
        "coverage": s_coverage,
        "duration": s_total,
        "max_closure": s_max_closure,
        "tracker": perclos_tracker,
    }


print("run_realtime_scores() 준비 완료 (YuNet + PERCLOS)")

In [ ]:
# 눈을 감아 보거나 하품해 보면서 막대와 그래프가 움직이는지 확인하세요.
# 정지: 셀 왼쪽 ■ (interrupt) 버튼
#
# PERCLOS 창이 60초라 그만큼은 돌아야 값이 찹니다 (그 전에는 warming up).

summary = run_realtime_scores(max_seconds=180, threshold=0.6,
                              perclos_window=60.0, perclos_threshold=0.15)

# 더 빠르게 (검출 4.6ms): run_realtime_scores(max_seconds=180, detect_width=320)
# 어두워서 느리면      : run_realtime_scores(max_seconds=180, manual_exposure=True)

---

## 잘 안 될 때

| 증상 | 원인 |
|---|---|
| 입을 다물어도 YAWN 이 뜬다 | `face_landmarker.task` 가 없어 게이트가 꺼진 것. 2번 셀 출력 확인 |
| 게이트 임계값 경고가 뜬다 | 가중치와 게이트가 다른 임계값. 둘을 맞추거나 데이터셋을 다시 만들 것 |
| NO FACE 가 자주 뜬다 | 너무 가까이 앉았을 수 있음. 640×480 에서 얼굴 높이가 264px 을 넘으면 crop 이 학습보다 타이트해진다 |
| 느리다 | `run_realtime_scores(detect_width=320)`. 어두우면 `manual_exposure=True` |
| 말하기에 경보가 뜬다 | `src/yawn_accumulator.py` 의 `DEFAULT_FIRE` 를 올린다 |
| 하품을 놓친다 | `YawnAccumulator(peak_min=0.0, fire=0.20)` |

## 알려진 한계

**손으로 가린 하품은 거의 못 잡습니다 (Recall 0.162).** 랜드마커가 손 뒤의 입을
"다물었다" 고 재서 게이트가 79% 를 끊습니다. 손-입 겹침을 따로 검출하는 경로가
필요하고, 임계값 조정으로는 해결되지 않습니다.

성능 근거와 학습 과정은 `model/artifacts/README.md` 와 저장소 `README.md` 를 볼 것.
